# TP3 — Fine-tuning Gemma 3 (4B) with LoRA on a Khmer Banking FAQ dataset

## 1. Installation

In [ ]:
import os

if "COLAB_" in "".join(os.environ):
    !pip install -q --upgrade unsloth unsloth_zoo
else:
    !pip install -q unsloth
!pip install -q sacrebleu

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Configuration

In [ ]:
CONFIG = dict(
    model_name="unsloth/gemma-3-4b-it",
    max_seq_length=2048,
    load_in_4bit=True,

    r=16,
    lora_alpha=16,
    lora_dropout=0,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_steps=5,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    optim="adamw_8bit",
    seed=3407,

    val_fraction=0.10,
    eval_steps=10,
    load_best_model_at_end=True,
    n_eval_generate=15,
)

DATA_PATH = "dataasilida_cleaned.csv"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## 3. Load Gemma 3 (4B) in 4-bit and attach LoRA adapters

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template


def build_model(cfg):
    model, tokenizer = FastModel.from_pretrained(
        model_name=cfg["model_name"],
        max_seq_length=cfg["max_seq_length"],
        load_in_4bit=cfg["load_in_4bit"],
        load_in_8bit=False,
        full_finetuning=False,
    )
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=cfg["r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        bias="none",
        random_state=cfg["seed"],
    )
    tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
    return model, tokenizer


model, tokenizer = build_model(CONFIG)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable LoRA parameters: {trainable:,} ({100 * trainable / total:.2f}% of {total:,})")

## 4. Dataset preparation

### 4.1 Load and repair the CSV

In [ ]:
import csv
import random
import re

import pandas as pd

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print(f"'{DATA_PATH}' not found -> please upload it")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Put {DATA_PATH} next to this notebook")


def load_and_repair(path):
    with open(path, encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))

    records, repaired, dropped = [], 0, 0
    for row in rows[1:]:
        if len(row) == 2:
            question, answer = row
        elif len(row) == 1 and "?" in row[0]:
            question, answer = row[0].split("?", 1)
            question, answer = question + "?", answer.lstrip(" ,")
            repaired += 1
        else:
            dropped += 1
            continue
        records.append({"input": question, "output": answer})
    return pd.DataFrame(records), repaired, dropped


raw_df, n_repaired, n_dropped = load_and_repair(DATA_PATH)
print(f"Rows read: {len(raw_df)} | repaired: {n_repaired} | dropped: {n_dropped}")

### 4.2 Clean the text

In [ ]:
def clean_text(text):
    text = str(text).replace("\\n", "\n").replace("​", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


df = raw_df.copy()
df["input"] = df["input"].map(clean_text)
df["output"] = df["output"].map(clean_text)

n_before = len(df)
df = df[(df.input.str.len() > 0) & (df.output.str.len() > 0)]
df = df.drop_duplicates(subset=["input", "output"]).reset_index(drop=True)
print(f"After cleaning: {len(df)} rows (removed {n_before - len(df)})")
print(f"Rows sharing a question with another row: {df.input.duplicated(keep=False).sum()}")

display(pd.DataFrame({"question": df.input.str.len().describe(),
                      "answer": df.output.str.len().describe()}).round(0))
df.sample(3, random_state=1)

### 4.3 Convert to the Gemma-3 chat format

In [ ]:
def to_chat_text(question, answer):
    messages = [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return text.removeprefix("<bos>")


df["text"] = [to_chat_text(q, a) for q, a in zip(df.input, df.output)]
print(df.text.iloc[0])

### 4.4 Check token lengths

In [ ]:
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

df["n_tokens"] = [len(text_tokenizer(t, add_special_tokens=True)["input_ids"]) for t in df.text]
display(df.n_tokens.describe().round(0).to_frame("tokens per example"))

too_long = (df.n_tokens > CONFIG["max_seq_length"]).sum()
print(f"Examples longer than max_seq_length={CONFIG['max_seq_length']}: {too_long}")

df = df[df.n_tokens <= CONFIG["max_seq_length"]].reset_index(drop=True)
chars = (df.input.str.len() + df.output.str.len()).sum()
print("Chars per token (avg):", round(chars / df.n_tokens.sum(), 2))

### 4.5 Train / validation split — grouped by question

In [ ]:
from datasets import Dataset

rng = random.Random(CONFIG["seed"])
questions = sorted(df.input.unique())
rng.shuffle(questions)

n_val = max(1, round(CONFIG["val_fraction"] * len(questions)))
val_questions = set(questions[:n_val])

train_df = df[~df.input.isin(val_questions)].reset_index(drop=True)
val_df = df[df.input.isin(val_questions)].reset_index(drop=True)
assert not set(train_df.input) & set(val_df.input), "train/validation leakage"

train_dataset = Dataset.from_pandas(train_df[["text"]])
val_dataset = Dataset.from_pandas(val_df[["text"]])

effective_batch = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]
steps_per_epoch = -(-len(train_df) // effective_batch)
print(f"Train: {len(train_df)} examples | Validation: {len(val_df)} examples")
print(f"Steps per epoch: {steps_per_epoch} | Total steps: {steps_per_epoch * CONFIG['num_train_epochs']}")

## 5. Baseline — answers before fine-tuning

In [ ]:
def generate_answer(model, question, max_new_tokens=300, sample=False):
    messages = [{"role": "user", "content": [{"type": "text", "text": question}]}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    ).removeprefix("<bos>")
    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")

    gen_kwargs = dict(max_new_tokens=max_new_tokens, use_cache=True)
    if sample:
        gen_kwargs.update(do_sample=True, temperature=1.0, top_p=0.95, top_k=64)
    else:
        gen_kwargs.update(do_sample=False)

    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


eval_sample = val_df.sample(
    min(CONFIG["n_eval_generate"], len(val_df)), random_state=CONFIG["seed"]
).reset_index(drop=True)

FastModel.for_inference(model)
baseline_answers = []
for i, question in enumerate(eval_sample.input, start=1):
    baseline_answers.append(generate_answer(model, question))
    print(f"[{i}/{len(eval_sample)}] done")
eval_sample["before_finetune"] = baseline_answers
FastModel.for_training(model)

print("\nQ:", eval_sample.input[0])
print("Reference:", eval_sample.output[0][:300])
print("Base model:", baseline_answers[0][:300])

## 6. Build the trainer — SFT with a validation set

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only


def build_trainer(model, cfg, output_dir):
    args = SFTConfig(
        output_dir=output_dir,
        dataset_text_field="text",
        per_device_train_batch_size=cfg["per_device_train_batch_size"],
        per_device_eval_batch_size=cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
        num_train_epochs=cfg["num_train_epochs"],
        warmup_steps=cfg["warmup_steps"],
        learning_rate=cfg["learning_rate"],
        lr_scheduler_type=cfg["lr_scheduler_type"],
        weight_decay=cfg["weight_decay"],
        optim=cfg["optim"],
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=cfg["eval_steps"],
        save_strategy="steps",
        save_steps=cfg["eval_steps"],
        save_total_limit=2,
        save_only_model=True,
        load_best_model_at_end=cfg["load_best_model_at_end"],
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        seed=cfg["seed"],
        report_to="none",
        dataset_num_proc=2,
        packing=False,
    )
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        args=args,
    )
    return train_on_responses_only(
        trainer,
        instruction_part="<start_of_turn>user\n",
        response_part="<start_of_turn>model\n",
    )


trainer = build_trainer(model, CONFIG, output_dir="outputs_exp1")

**Verify the masking**

In [ ]:
pad_id = text_tokenizer.pad_token_id


def show_masking(dataset, idx=0):
    input_ids = dataset[idx]["input_ids"]
    labels = dataset[idx]["labels"]
    kept = [t if label != -100 else pad_id for t, label in zip(input_ids, labels)]
    n_masked = sum(label == -100 for label in labels)

    print("FULL INPUT :", text_tokenizer.decode(input_ids)[:400])
    print("LOSS TOKENS:", text_tokenizer.decode(kept).replace(text_tokenizer.pad_token, "").strip()[:400])
    print(f"-> {n_masked}/{len(labels)} tokens masked\n")


print("=== train ===")
show_masking(trainer.train_dataset)
print("=== validation ===")
show_masking(trainer.eval_dataset)

## 7. Train (Experiment 1 — main run)

In [ ]:
gpu = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_mem = round(gpu.total_memory / 1024**3, 3)
print(f"GPU = {gpu.name}. Max memory = {max_mem} GB. Reserved before training = {start_mem} GB.")

trainer_stats = trainer.train()

used_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\nTraining time: {trainer_stats.metrics['train_runtime'] / 60:.1f} min")
print(f"Peak reserved memory = {used_mem} GB ({100 * used_mem / max_mem:.1f}% of GPU); "
      f"used for training = {used_mem - start_mem:.2f} GB")
print("Best checkpoint:", trainer.state.best_model_checkpoint,
      "| best validation loss:", trainer.state.best_metric)

## 8. Loss analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def split_history(log_history):
    train_log = pd.DataFrame(
        [h for h in log_history if "loss" in h and "eval_loss" not in h]
    )[["step", "epoch", "loss"]]
    eval_log = pd.DataFrame(
        [h for h in log_history if "eval_loss" in h]
    )[["step", "epoch", "eval_loss"]]
    train_log["loss_smooth"] = train_log["loss"].rolling(10, min_periods=1).mean()
    return train_log, eval_log


def diagnose(train_log, eval_log, rise_tol=0.05):
    best_i = eval_log.eval_loss.idxmin()
    best_step = int(eval_log.step[best_i])
    best_loss = eval_log.eval_loss[best_i]
    final_val = eval_log.eval_loss.iloc[-1]
    final_train = train_log.loss_smooth.iloc[-1]
    rise = (final_val - best_loss) / best_loss

    last = eval_log.eval_loss.tail(3).values
    still_falling = len(last) == 3 and last[0] > last[1] > last[2]

    if rise > rise_tol:
        verdict = (f"OVERFITTING: validation loss was lowest at step {best_step} ({best_loss:.3f}) "
                   f"and then rose {100 * rise:.0f}% to {final_val:.3f} while training loss kept "
                   f"falling to {final_train:.3f}. Training after ~step {best_step} memorises the "
                   f"training answers instead of generalising.")
    elif still_falling:
        verdict = (f"UNDERFITTING / not converged: validation loss is still decreasing at the end "
                   f"({final_val:.3f}). Training longer (or a higher learning rate) should still help.")
    else:
        verdict = (f"GOOD FIT: validation loss flattened around {best_loss:.3f} "
                   f"(best at step {best_step}) with a train/val gap of {final_val - final_train:.3f}.")

    return dict(
        best_step=best_step,
        best_val_loss=round(best_loss, 4),
        final_val_loss=round(final_val, 4),
        final_train_loss_smoothed=round(final_train, 4),
        gap=round(final_val - final_train, 4),
        val_rise_after_best_pct=round(100 * rise, 1),
        verdict=verdict,
    )


def plot_losses(train_log, eval_log, title, path):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_log.step, train_log.loss, color="tab:blue", alpha=0.25, lw=1,
            label="Training loss (per step)")
    ax.plot(train_log.step, train_log.loss_smooth, color="tab:blue", lw=2,
            label="Training loss (10-step average)")
    ax.plot(eval_log.step, eval_log.eval_loss, "o-", color="tab:orange", lw=2,
            label="Validation loss")

    best = eval_log.eval_loss.idxmin()
    ax.axvline(eval_log.step[best], color="tab:green", ls="--", lw=1.5,
               label=f"Best checkpoint (step {int(eval_log.step[best])})")

    for epoch in range(1, int(np.ceil(train_log.epoch.max()))):
        step = train_log.loc[(train_log.epoch - epoch).abs().idxmin(), "step"]
        ax.axvline(step, color="gray", ls=":", lw=1)
        ax.text(step, ax.get_ylim()[1] * 0.95, f" epoch {epoch}", color="gray")

    ax.set(xlabel="Training step", ylabel="Cross-entropy loss", title=title)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()


train_hist, eval_hist = split_history(trainer.state.log_history)
train_hist.to_csv(f"{RESULTS_DIR}/exp1_train_loss.csv", index=False)
eval_hist.to_csv(f"{RESULTS_DIR}/exp1_val_loss.csv", index=False)

plot_losses(train_hist, eval_hist,
            "Experiment 1 — Gemma 3 4B + LoRA on Khmer Banking FAQ",
            f"{RESULTS_DIR}/exp1_loss_curve.png")

diag1 = diagnose(train_hist, eval_hist)
print(diag1["verdict"])
pd.Series(diag1).drop("verdict").to_frame("Experiment 1")

In [ ]:
eval_hist.assign(
    train_loss_smoothed=eval_hist.step.map(train_hist.set_index("step").loss_smooth),
    gap=lambda d: d.eval_loss - d.train_loss_smoothed,
).round(4)

## 9. Compare results

### 9.1 New dataset vs original dataset — training-loss curves

In [ ]:
ORIGINAL_DATA_KH_LOSS = [
    5.3791, 6.0100, 6.1185, 4.9861, 4.9263, 4.6684, 3.9622, 3.9256, 3.0602, 2.8922,
    2.9186, 2.3912, 2.3802, 2.0256, 1.9073, 1.9099, 1.4648, 1.4912, 1.2801, 1.3175,
    1.2119, 1.2252, 0.9541, 1.1308, 0.9628, 1.0070, 0.9160, 0.8377, 0.8726, 0.8572,
    0.8951, 0.8677, 0.8285, 0.6928, 0.7205, 0.8073, 0.7801, 0.7268, 0.7975, 0.6175,
    0.5736, 0.6828, 0.6521, 0.6318, 0.5716, 0.6379, 0.5190, 0.5130, 0.5303, 0.4682,
    0.5486, 0.5945, 0.5055, 0.4909, 0.5126, 0.4564, 0.4642, 0.4490, 0.4791, 0.5751,
]
orig = pd.DataFrame({"step": range(1, 61), "loss": ORIGINAL_DATA_KH_LOSS})

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(orig.step, orig.loss, color="tab:purple", lw=2,
        label="Original: data_kh.csv (robot commands → JSON), train loss")
ax.plot(train_hist.step, train_hist.loss_smooth, color="tab:blue", lw=2,
        label="New: Banking FAQ, train loss (10-step avg)")
ax.plot(eval_hist.step, eval_hist.eval_loss, "o-", color="tab:orange",
        label="New: Banking FAQ, validation loss")
ax.set(xlabel="Training step", ylabel="Loss", title="Original dataset vs new dataset")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/dataset_comparison.png", dpi=150)
plt.show()

comparison = pd.DataFrame({
    "Original (data_kh.csv)": {
        "task": "Khmer command → JSON action",
        "train examples": 2930,
        "LoRA r / alpha": "8 / 8",
        "training": "60 steps (16% of 1 epoch)",
        "validation set": "none",
        "loss @ step 60": orig.loss.tail(5).mean().round(3),
        "best val loss": "n/a",
    },
    "New (Banking FAQ)": {
        "task": "Khmer banking question → answer",
        "train examples": len(train_df),
        "LoRA r / alpha": f"{CONFIG['r']} / {CONFIG['lora_alpha']}",
        "training": f"{CONFIG['num_train_epochs']} epochs ({int(train_hist.step.max())} steps)",
        "validation set": f"{len(val_df)} examples",
        "loss @ step 60": train_hist.loc[train_hist.step <= 60, "loss"].tail(5).mean().round(3),
        "best val loss": diag1["best_val_loss"],
    },
})
comparison.to_csv(f"{RESULTS_DIR}/dataset_comparison.csv")
comparison

### 9.2 Before vs after fine-tuning — answer quality on unseen validation questions

In [ ]:
from sacrebleu.metrics import CHRF

chrf = CHRF()

FastModel.for_inference(model)
after_answers = []
for i, question in enumerate(eval_sample.input, start=1):
    after_answers.append(generate_answer(model, question))
    print(f"[{i}/{len(eval_sample)}] done")
eval_sample["after_finetune"] = after_answers

references = list(eval_sample.output)
eval_sample["chrF_before"] = [chrf.sentence_score(h, [r]).score
                              for h, r in zip(eval_sample.before_finetune, references)]
eval_sample["chrF_after"] = [chrf.sentence_score(h, [r]).score
                             for h, r in zip(eval_sample.after_finetune, references)]

corpus_before = chrf.corpus_score(list(eval_sample.before_finetune), [references]).score
corpus_after = chrf.corpus_score(list(eval_sample.after_finetune), [references]).score
print(f"\nCorpus chrF on {len(eval_sample)} unseen validation questions: "
      f"before = {corpus_before:.1f}   after = {corpus_after:.1f}")
print(f"Questions improved: {(eval_sample.chrF_after > eval_sample.chrF_before).sum()} / {len(eval_sample)}")
eval_sample.to_csv(f"{RESULTS_DIR}/before_after_answers.csv", index=False)

pd.set_option("display.max_colwidth", 200)
eval_sample[["input", "output", "before_finetune", "after_finetune", "chrF_before", "chrF_after"]].round(1)

In [ ]:
for i in range(min(3, len(eval_sample))):
    row = eval_sample.iloc[i]
    print(f"━━━━━━━━ Example {i + 1} ━━━━━━━━")
    print(f"QUESTION : {row.input}")
    print(f"REFERENCE: {row.output[:350]}")
    print(f"BEFORE   : {row.before_finetune[:350]}")
    print(f"AFTER    : {row.after_finetune[:350]}")
    print(f"chrF before {row.chrF_before:.1f} -> after {row.chrF_after:.1f}\n")

### 9.3 Try your own question (live demo)

In [ ]:
from transformers import TextStreamer

question = "តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា មានសុពលភាពប៉ុន្មានឆ្នាំ"

messages = [{"role": "user", "content": [{"type": "text", "text": question}]}]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
).removeprefix("<bos>")
inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")

_ = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=True,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
    streamer=TextStreamer(text_tokenizer, skip_prompt=True, skip_special_tokens=True),
)

## 10. Save the fine-tuned LoRA adapter

In [ ]:
model.save_pretrained("gemma3_banking_lora")
tokenizer.save_pretrained("gemma3_banking_lora")
print("Saved to ./gemma3_banking_lora")

## 11. (Optional) Experiment 2 — hyperparameter comparison

In [ ]:
import gc

RUN_EXPERIMENT_2 = False

if RUN_EXPERIMENT_2:
    CONFIG_2 = {**CONFIG, "learning_rate": 2e-5}

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = build_model(CONFIG_2)
    trainer = build_trainer(model, CONFIG_2, output_dir="outputs_exp2")
    trainer.train()

    train_hist2, eval_hist2 = split_history(trainer.state.log_history)
    train_hist2.to_csv(f"{RESULTS_DIR}/exp2_train_loss.csv", index=False)
    eval_hist2.to_csv(f"{RESULTS_DIR}/exp2_val_loss.csv", index=False)
    diag2 = diagnose(train_hist2, eval_hist2)
    print(diag2["verdict"])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(train_hist.step, train_hist.loss_smooth, color="tab:blue", lw=2,
            label="Exp 1 (lr=2e-4) train")
    ax.plot(eval_hist.step, eval_hist.eval_loss, "o-", color="tab:blue", alpha=0.6,
            label="Exp 1 (lr=2e-4) validation")
    ax.plot(train_hist2.step, train_hist2.loss_smooth, color="tab:red", lw=2,
            label="Exp 2 (lr=2e-5) train")
    ax.plot(eval_hist2.step, eval_hist2.eval_loss, "s-", color="tab:red", alpha=0.6,
            label="Exp 2 (lr=2e-5) validation")
    ax.set(xlabel="Training step", ylabel="Loss", title="Effect of learning rate")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/exp1_vs_exp2_learning_rate.png", dpi=150)
    plt.show()

    display(pd.DataFrame({"Exp 1 (lr=2e-4)": diag1, "Exp 2 (lr=2e-5)": diag2}).drop("verdict"))

## 12. Download all results (plots + CSVs)

In [ ]:
import json
import shutil

with open(f"{RESULTS_DIR}/config_exp1.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
with open(f"{RESULTS_DIR}/diagnosis_exp1.json", "w") as f:
    json.dump(diag1, f, indent=2, ensure_ascii=False)

shutil.make_archive("tp3_results", "zip", RESULTS_DIR)
print(sorted(os.listdir(RESULTS_DIR)))

try:
    from google.colab import files
    files.download("tp3_results.zip")
except ImportError:
    print("Saved tp3_results.zip")